# Learning-Map Tour --- ELF: Embedded Language Flows

Hu et al. 2026, arXiv:2605.10938. Notebook companion to `tour.md` / `tour.tex`.
v1.9 per-paper-tour rollout artefact. Last updated 2026-05-15.


## 1. Reader's Contract

**Audience.** Math-grad to ML researcher.  
**Background.** Walk [math-foundations](https://github.com/pleyva2004/math-foundations) at the *Graduate* entry point first.  
**Expected reading time.** ~3-4 hours (30m foundations refresh, 90m math deep-dive, 30m improvements, 60m hands-on).  
**Elevator.** ELF runs continuous-time flow matching in a frozen pretrained embedding space and reuses the same network's final timestep as the decoder via the weight-tied unembedding matrix.


## 2. Foundations Walk

The cell below loads `notation.json` (if present in the paper's sandbox) and prints a navigation summary keyed against the math-foundations concept slugs needed to read ELF in full.


In [ ]:
# Notation lookup + foundations navigation summary.
import json, os

NOTATION_PATH = '../sandbox/notation.json'
FOUNDATIONS = [
    ('06', 'linear-maps',                 'skim',  'Embedding matrix W_E and its transpose; weight-tying.'),
    ('14', 'gradient-jacobian',           'read',  'Velocity-field regression target; gradient flow on FM loss.'),
    ('24', 'pdf',                          'read',  'z_0 ~ N(0,I), z_1 ~ p_data are densities on R^d.'),
    ('25', 'expectation',                  'skim',  'Flow-matching loss is an expectation over (t, z_0, z_1).'),
    ('28', 'change-of-variables-probability','drill','Continuity equation underlies flow matching.'),
    ('29', 'ode',                          'read',  'Inference is the ODE dz/dt = v_theta integrated 0 -> 1.'),
    ('32', 'brownian-motion',              'skim',  'Lineage: continuous-time noise diffusion is built on Wiener.'),
    ('33', 'sde',                          'skim',  'DDPM / SEDD are SDEs; flow matching is the deterministic limit.'),
    ('37', 'cross-entropy',                'read',  'Final-step loss: CE(W_E^T z_1, true token).'),
    ('38', 'kl-divergence',                'skim',  'CFG re-derives as KL-regularised posterior interpolation.'),
    ('40', 'information-geometry',         'drill', 'Curved-path improvement uses Fisher metric on embeddings.'),
    ('41', 'optimal-transport',            'drill', 'Linear interp = Euclidean W_2 geodesic; OT geodesic may differ.'),
]

if os.path.exists(NOTATION_PATH):
    with open(NOTATION_PATH) as f:
        notation = json.load(f)
    print(f"notation.json: {len(notation)} entries loaded from sandbox.")
else:
    notation = {}
    print('notation.json not found in sandbox (expected during early sandbox stages).')

print()
print(f"{'#':>3}  {'pacing':<6}  slug")
print('-' * 60)
for nid, slug, pace, _ in FOUNDATIONS:
    print(f'{nid:>3}  {pace:<6}  {slug}')
print()
print(f'Foundations covered: {len(FOUNDATIONS)} concepts.')


## 3. Paper Concepts Walk

Eight ELF-specific moving parts (each tied to an equation or section in `02-math-deep-dive.md`):

1. **Linear interpolation path** --- $z_t = (1-t) z_0 + t z_1$ (Eq. 1)
2. **Velocity-field prediction** --- regress against $z_1 - z_0$ (Eq. 2)
3. **Weight-tied unembedding** --- logits $= W_E^\top z_1$ at $t = 1$
4. **Classifier-free guidance** --- $v_\text{cfg} = \omega v_\theta(\cdot|c) + (1-\omega) v_\theta(\cdot|\emptyset)$ (Eq. 4)
5. **Self-conditioning** --- $c$ is the model's own intermediate prediction
6. **Training-time CFG** --- single forward pass outputs $v_\text{cfg}$ directly
7. **Pretrained contextual embeddings** --- $W_E$ is frozen, BERT-style
8. **Latent-diffusion ancestry** --- ELF $=$ LDM with encoder $=$ embedding matrix, decoder $=$ its transpose


## 4. Improvements Walk

Per `05-improvements.tex`. Each is a **PROOF** (`.tex`) or a **MEASUREMENT** (Python).

| # | Type | Title | Artefact | Status |
|---|------|-------|----------|--------|
| Math 1 | PROOF | When is the linear path optimal? | `proofs/curved-path-optimality.tex` | live |
| Math 2 | MEASUREMENT | Embedding-quality vs flow contributions | counterfactual run | deferred |
| Code 1 | MEASUREMENT | Scheduled CFG | `improvements/cfg-schedule.py` | live |
| Code 2 | MEASUREMENT | Multimodal embedding sharing prototype | `improvements/multimodal-embed-share.py` | NEW sketch |
| Exp 1  | MEASUREMENT | AR baseline at matched compute | 105M AR run | deferred |
| Exp 2  | MEASUREMENT | Multimodal embedding sharing experiment | `improvements/multimodal-embed-share.py` | live (sketch) |
| Theory 1 | PROOF | ELF as Latent Diffusion w/ weight-tied decoder | `proofs/elf-as-LDM.tex` | live |
| Theory 2 | PROOF | ELF as discrete information bottleneck | `proofs/elf-info-bottleneck.tex` | deferred |


In [ ]:
# Measurement run #1 --- scheduled CFG (Code Improvement 1).
# Toy ELF on a 2D Gaussian mixture; compares fixed-omega to U-shape and linear-ramp schedules.
# This will execute against `improvements/cfg-schedule.py` once that file exposes a measure() entry point.
import sys, os
sys.path.append(os.path.abspath('../improvements'))

try:
    from cfg_schedule import measure
    print(measure())
except Exception as e:
    print(f'cfg_schedule.measure() not yet available: {type(e).__name__}: {e}')
    print('Expected output (per 05-improvements.tex):')
    print('  fixed omega = 3.0          : log p = -2.41,  entropy = 1.87')
    print('  U-shape (beta=0.5)         : log p = -1.93,  entropy = 1.92')
    print('  linear ramp omega = 1+4t   : log p = -2.18,  entropy = 1.85')


In [ ]:
# Measurement run #2 --- multimodal embedding sharing (Code Improvement 2 / Experimental 2).
# Sketch prototype: shared embedding space across text and image, single flow-matching network.
import sys, os
sys.path.append(os.path.abspath('../improvements'))

try:
    from multimodal_embed_share import measure
    print(measure())
except Exception as e:
    print(f'multimodal_embed_share.measure() not yet available: {type(e).__name__}: {e}')
    print('Sketch only --- documents the experimental design and a tiny synthetic-data run.')


## 5. What To Do Next

1. **Most promising proposal --- scheduled CFG.** No retraining required; only inference-time modification.
   Email the ELF authors with the schedule + the toy result.
2. **Single most valuable experiment --- AR baseline at matched compute.** Train a 105M decoder-only    Transformer on the same OpenWebText for the same token budget. Adds the missing row of Table 1.
3. **Follow-on paper direction --- *How much of ELF is the manifold, how much is the flow?***    Decompose the lift and publish the right CFG schedule for both regimes.


---
*Tour notebook (v1.9). Companion to `tour.md` and `tour.tex`. Re-run cells after `cfg-schedule.py` and `multimodal-embed-share.py` expose their `measure()` entry points.*
